# Direct-to-Lakebase Realtime Kafka Ingestion

This example notebook uses Spark Structured Streaming's [real time mode](https://docs.databricks.com/aws/en/structured-streaming/real-time/concepts) to ingest data from Kafka and land it immediately in Lakebase, with subsecond latency. It uses [lakebase-foreachwriter](https://github.com/christophergrant/lakebase-foreachwriter) to write batches of rows into your target Lakebase table easily and automatically.

## Requirements
* You must set `spark.databricks.streaming.realTimeMode.enabled true` in your spark configs.
* You must use a cluster running in single-user mode with at least 10 total vCPUs.
* You must have a volume set up to use as a checkpoint location.
* You must create your target table in Lakebase with an appropriate schema.
* Either install lakebase-foreachwriter as a wheel, or place its `LakebaseForeachWriter.py` file alongside this notebook.

## Other Infrastructure
This notebook uses AWS Managed Streaming for Apache Kafka (MSK) as a Kafka broker. Cell 3 is for AWS setup and requires AWS credentials, as does cell 4 for Kafka. If you don't use AWS MFK, see our documentation on [Kafka authentication from Databricks](https://docs.databricks.com/aws/en/connect/streaming/kafka/authentication) to understand your options.

In [ ]:
%pip install psycopg

In [ ]:
# Authentication: change this to use your own service credential if using MSK, otherwise see note above about authentication.
SERVICE_CREDENTIAL_NAME = "<your-service-credential>"

# Lakebase
LAKEBASE_BRANCH = "projects/<your-project>/branches/<your-branch>"
LAKEBASE_TABLE = "public.kafka"
USERNAME = "<your-databricks-username>"  # Postgres role: your Databricks user email, or the app service principal's application id

# Other
CHECKPOINT_LOCATION = "/Volumes/<catalog>/<schema>/<volume>/kafka-checkpoint"

In [ ]:
import os
import boto3

AWS_REGION = "us-east-1"
CLUSTER_ARN = "arn:aws:kafka:<region>:<aws-account-id>:cluster/<cluster-name>/<cluster-uuid>"

os.environ["AWS_ACCESS_KEY_ID"] = dbutils.secrets.get(catalog="<your-secret-catalog>", schema="<your-secret-schema>", key="aws_access_key_id")
os.environ["AWS_SECRET_ACCESS_KEY"] = dbutils.secrets.get(catalog="<your-secret-catalog>", schema="<your-secret-schema>", key="aws_secret_access_key")

msk_client = boto3.client("kafka", region_name=AWS_REGION)
bootstrap = msk_client.get_bootstrap_brokers(ClusterArn=CLUSTER_ARN)["BootstrapBrokerStringPublicSaslIam"]

In [ ]:
kafka_options = {
    "kafka.bootstrap.servers": bootstrap,
    "subscribe": "lakebase-test",
    "databricks.serviceCredential": SERVICE_CREDENTIAL_NAME,
}

In [ ]:
from databricks.sdk import WorkspaceClient
from LakebaseForeachWriter import LakebaseForeachWriter

# Use the Lakebase SDK to get the list of endpoints from our preferred Lakebase branch, then select the primary read-write endpoint.
# We'll generate the oauth token ourselves the first time out, but after initialization the Lakebase foreachwriter will handle token refreshes.
w = WorkspaceClient()
endpoints = list(w.postgres.list_endpoints(parent=LAKEBASE_BRANCH))
endpoint = endpoints[0]
oauth_token = w.postgres.generate_database_credential(endpoint=endpoint.name).token

In [ ]:
from pyspark.sql.functions import from_json, col, unbase64, from_json

# Read from Kafka and preprocess data.
df = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .load()

    # Data transformations specific to our data - the input is a simple JSON with an "id" field and a "value" field. By default
    # it's read as binary, so cast it and parse it from JSON with a known schema.
    .select(from_json(col("value").cast("STRING"), "STRUCT<id:STRING, value:STRING>").alias("value"))
    .select("value.*")
)

# Build the Lakebase writer; has to be done after the read query is created so that the writer
# knows the schema of the data it'll be receiving.
lakebase_writer = LakebaseForeachWriter(
    username=USERNAME,
    password=oauth_token,
    table=LAKEBASE_TABLE,
    df=df,
    lakebase_name=LAKEBASE_BRANCH.split("/")[1],
    mode="insert"
)

# Now write to Lakebase directly using the writer. The writer will take care of connection maintenance and such.
(
    df.writeStream
    .outputMode("update")
    .foreach(lakebase_writer)

    # This is the magic one-liner that makes realtime mode work. Make sure you also have this in your spark configs:
    # spark.databricks.streaming.realTimeMode.enabled true
    .trigger(realTime="5 minutes")
    .option("checkpointLocation", CHECKPOINT_LOCATION)
    .start()
)